# 17 · Production smoke: S3 scratch and the Left side

Tests 0-3 of `docs/plans/prod_smoke_tests.md`, run by hand against your **real** pairs config and real Left databases. It proves the S3 scratch prefix is writable, every Left source answers a preflight, the Left side has rows in a recent window, and a one-day pull returns exactly the rows that day was counted with.

Everything is read-only on the Left side. The only writes are the S3 scratch prefix (removed at the end) and pulled CSVs under `WORKDIR` (also removed unless `KEEP=True`). Each pair is checked on its own; a failing pair is recorded and the loop moves on.

## How to approach

- **Prerequisites**: kernel = the repo `.venv`; `uv sync` plus `--extra athena` / `--extra ldap` if any pair needs them (a missing extra shows up as a preflight error naming it). Left credentials (`<macro>_USR`, `_PWD`, `_DSN`, ...) live in the project `.env` (`ENV_FILE`). AWS credentials for `S3_SCRATCH` come from your usual AWS profile / env (`AWS_PROFILE`, `DTRACK_S3_ENDPOINT_URL` if you use one).
- **Run order**: top to bottom. Test 0 and 1 are fast. Test 2 runs one `GROUP BY` count per pair. Test 3 pulls one full day per pair (skipped above `MAX_ROWS`), so it is the slowest.
- **What to look at**: the summary table after each test (`ok` column), the per-day counts table, and `match` in test 3.
- **Failure modes**: preflight `ok=False` = connection, credentials, table name or missing extra; no partitions in test 2 = wrong `date_col` / `date_type` / `where`, or the window is before the data; a test 3 mismatch = the count and the pull disagree on the date predicate (date type or timezone).

In [ ]:
import os
from pathlib import Path

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")   # your real pairs config
ENV_FILE = os.environ.get("DTRACK_ENV_FILE", "C:/work/dtrack/.env")             # project .env with <macro>_USR / _PWD / _DSN
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")                           # used when PAIRS = [PAIR]
PAIRS = None                     # None = every active (non-skip) pair; or ["orders", "cards"]
TO_DATE = os.environ.get("DTRACK_PROD_TO") or None      # None = yesterday; or "YYYY-MM-DD"
FROM_DATE = os.environ.get("DTRACK_PROD_FROM") or None  # None = TO_DATE - (DAYS - 1)
DAYS = int(os.environ.get("DTRACK_PROD_DAYS", "3"))     # window length in days
MAX_ROWS = int(os.environ.get("DTRACK_PROD_MAX_ROWS", "200000"))   # skip the one-day pull above this
S3_SCRATCH = os.environ.get("DTRACK_PROD_S3", "s3://<bucket>/dtrack/nb-scratch")   # only S3 write location
WORKDIR = Path.home() / ".local/plans/nbs/_work/17"     # pulled CSVs land here
KEEP = False                     # True = keep the S3 scratch objects and pulled CSVs

In [ ]:
# Load the project .env first: load_config() only searches from the current
# directory, and this notebook does not run from the repo root.
import copy
import json
import traceback
from datetime import date, timedelta

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from dtrack_left.cli.config import load_config

pd.set_option("display.max_colwidth", 120)
if Path(ENV_FILE).exists():
    load_dotenv(ENV_FILE, override=False)
else:
    print(f"warning: {ENV_FILE} not found; Left credentials must already be in the environment")

# The real config: connection profiles resolved, col_map files expanded, validated.
config = load_config(CONFIG_PATH)

# Which pairs: every active (non-skip) pair, or the PAIRS list.
active = [name for name, spec in config["pairs"].items() if not spec.get("skip")]
if PAIRS is None:
    names = active
else:
    unknown = [name for name in PAIRS if name not in config["pairs"]]
    skipped = [name for name in PAIRS if name in config["pairs"] and name not in active]
    if unknown:
        print("not in the config, ignored:", unknown)
    if skipped:
        print("marked skip in the config, ignored:", skipped)
    names = [name for name in PAIRS if name in active]

# The check window: the last DAYS days up to TO_DATE (default yesterday).
TO = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
FROM = FROM_DATE or (date.fromisoformat(TO) - timedelta(days=DAYS - 1)).isoformat()


# The pair exactly as configured, with fromDate/toDate narrowed to the window
# (a deep copy, so the loaded config is never changed).
def windowed(name):
    spec = copy.deepcopy(config["pairs"][name])
    spec["fromDate"], spec["toDate"] = FROM, TO
    return spec


WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"window {FROM}..{TO}; {len(names)} pair(s): {names}")

# A per-session tag so two runs never share a scratch folder.
from datetime import datetime

TAG = datetime.now().strftime("nb17-%Y%m%d-%H%M%S")
SCRATCH = f"{S3_SCRATCH.rstrip('/')}/{TAG}"
print("scratch prefix:", SCRATCH)

## Test 0 · S3 scratch round trip

`S3Store` (the same class the distributed runs use) writes one JSON probe, reads it back and lists the prefix. If this fails, the run folders and report backups will fail too.

In [ ]:
# Write, read back, list: the three calls every run makes against S3.
from dtrack_left.backends.store import S3Store

probe = {"ok": True, "tag": TAG}
try:
    store = S3Store(f"{SCRATCH}/s3-check")
    written = store.put_json("probe.json", probe)
    read_back = store.get_json("probe.json")
    listed = store.list_keys("")
    s3_check = pd.DataFrame([{
        "uri": store.root_uri,
        "written_key": written,
        "read_back_equal": read_back == probe,
        "listed": listed,
        "exists": store.exists("probe.json"),
        "ok": read_back == probe and listed == ["probe.json"],
    }])
except Exception as exc:
    s3_check = pd.DataFrame([{"uri": SCRATCH, "ok": False, "error": repr(exc)}])
display(s3_check)

## Test 1 · Left preflight

`source_from_config(pair["left"]).preflight()` runs `SELECT * FROM <relation> WHERE 1=0` (file sources also check that `date_col` exists). It proves connection, credentials and the table/query, without reading rows.

In [ ]:
# One preflight per pair; any exception (e.g. a missing driver) is recorded,
# not raised, so the remaining pairs still run.
from dtrack_left.backends.sources import source_from_config

SOURCES = {}
rows = []
for name in names:
    spec = windowed(name)
    try:
        source = source_from_config(spec["left"])
        SOURCES[name] = source
        health = source.preflight()
        rows.append({"pair": name, "source": spec["left"].get("source"), "ok": bool(health.get("ok")),
                     "error": health.get("error", "")})
    except Exception as exc:
        rows.append({"pair": name, "source": spec["left"].get("source"), "ok": False, "error": repr(exc)})
preflight_left = pd.DataFrame(rows)
display(preflight_left)

## Test 2 · Left counts in the window

`partition_counts(FROM, TO)` is the Stage 1 Left count: one `GROUP BY` day over the window. A pair passes when it has at least one day and more than zero rows. The counts are kept in `LEFT_COUNTS` for test 3.

In [ ]:
# Count each pair once; keep the per-day counts for the one-day pull below.
LEFT_COUNTS = {}
rows = []
for name in names:
    if name not in SOURCES:
        rows.append({"pair": name, "ok": False, "error": "no source (see preflight)"})
        continue
    try:
        counts = SOURCES[name].partition_counts(FROM, TO)
        LEFT_COUNTS[name] = counts
        rows.append({"pair": name, "days": len(counts), "rows": sum(counts.values()),
                     "first_day": min(counts) if counts else None, "last_day": max(counts) if counts else None,
                     "ok": bool(counts) and sum(counts.values()) > 0, "error": ""})
    except Exception as exc:
        rows.append({"pair": name, "ok": False, "error": repr(exc)})
left_counts = pd.DataFrame(rows)
display(left_counts)

In [ ]:
# Per-day counts side by side (one column per pair), to spot missing or thin days.
if LEFT_COUNTS:
    per_day = pd.DataFrame(LEFT_COUNTS).sort_index()
    per_day.index.name = "day"
    display(per_day)

## Test 3 · Pull one day and match its count

The smallest day of the window is pulled with `pull_window` using every mapped Left column plus the keys, exactly like Stage 2 does. The pulled row count must equal the count from test 2. Days above `MAX_ROWS` are skipped.

In [ ]:
# Smallest day per pair -> pull_window -> read back as text -> compare with its count.
PULLED = {}
rows = []
for name in names:
    counts = LEFT_COUNTS.get(name)
    if not counts:
        rows.append({"pair": name, "status": "skip", "note": "no Left counts (see test 2)"})
        continue
    day, counted = min(counts.items(), key=lambda item: item[1])
    if counted > MAX_ROWS:
        rows.append({"pair": name, "day": day, "counted": counted, "status": "skip",
                     "note": f"above MAX_ROWS={MAX_ROWS}"})
        continue
    spec = windowed(name)
    date_col = spec["left"]["date_col"].lower()
    columns = [column for column in spec["col_map"] if column.lower() != date_col]
    keys = [key.strip() for key in str(spec.get("key") or "").split(",") if key.strip()]
    output = WORKDIR / name / day / "left.csv"
    output.parent.mkdir(parents=True, exist_ok=True)
    try:
        path = SOURCES[name].pull_window([day], columns, output, key_columns=keys)
        frame = pd.read_csv(path, dtype=object)
        PULLED[name] = frame
        rows.append({"pair": name, "day": day, "counted": counted, "pulled": len(frame),
                     "match": len(frame) == counted, "columns": len(frame.columns),
                     "status": "ok" if len(frame) == counted else "MISMATCH", "note": str(path)})
    except Exception as exc:
        rows.append({"pair": name, "day": day, "counted": counted, "status": "error", "note": repr(exc)})
one_day = pd.DataFrame(rows)
display(one_day)

In [ ]:
# A peek at what was pulled: the first rows of each pair, all values as text
# (exactly how Stage 2 reads them before computing statistics).
for name, frame in PULLED.items():
    print(f"--- {name}: {len(frame)} rows x {len(frame.columns)} columns")
    display(frame.head(5))

## Cleanup

Removes the S3 scratch prefix and the pulled CSVs (production data on local disk) unless `KEEP=True`.

In [ ]:
# Delete everything under the scratch prefix unless KEEP=True. This is the only
# S3 location the notebook ever writes to.
import shutil

from dtrack_left.backends.store import S3Store

if KEEP:
    print("KEEP=True: left in place:", SCRATCH, "and", WORKDIR)
else:
    store = S3Store(SCRATCH)
    keys = store.list_keys("")
    for start in range(0, len(keys), 1000):
        batch = [{"Key": store._key(key)} for key in keys[start : start + 1000]]
        store.client.delete_objects(Bucket=store.bucket, Delete={"Objects": batch})
    print(f"deleted {len(keys)} object(s) under {SCRATCH}")
    shutil.rmtree(WORKDIR, ignore_errors=True)
    print("removed", WORKDIR)

## What to check

- Test 0: `ok=True`, `listed == ["probe.json"]`.
- Test 1: every pair `ok=True`; read the `error` text for the rest (driver, credentials, table name, missing extra).
- Test 2: every pair has `days > 0` and `rows > 0`; gaps in the per-day table are worth a look.
- Test 3: `match=True` for each pulled pair; a mismatch points at `date_col` / `date_type` / `where`.